# The goal here is to formalize the process of running a genome window, so that I can submit slurm scripts from this notebook.

In [76]:
import os
import subprocess
import numpy as np

# Start with outlier windows

In [52]:
windows_dir = "/n/holylfs05/LABS/hopkins_lab/Lab/PROJECTS/Phlox_assembly_collab/args_introgression/vcf_tarballs/results_20260210/results/outlier_windows/"
window_filenames = os.listdir(windows_dir)
window_filenames = [i for i in window_filenames if i.endswith('.gz')] # getting rid of index files
window_filenames = [i for i in window_filenames if 'rep1' in i] # my choice... only doing one rep per window for now

In [53]:
for window_filename in window_filenames:
    window_vcf_path = os.path.join(windows_dir,window_filename)
    
    results_dir = os.path.join('/n/home09/pfmckenzie/lab_lfs/Lab/PROJECTS/Phlox_assembly_collab/args_introgression/argweaver_analysis_feb2026/first_run/argweaver_runs/outlier_windows',window_filename)
    os.makedirs(results_dir,exist_ok=True)

    chrom, start, end = window_filename.split('.')[3].split('_')
    
    slurm_script = f"""#!/bin/sh
#SBATCH -c 4
#SBATCH -t 0-23:59:00
#SBATCH -p sapphire
#SBATCH --output={results_dir}/slurm-%j.out
#SBATCH --mem=20G
#SBATCH --job-name=aw

cd /n/home09/pfmckenzie/lab_lfs/Lab/PROJECTS/Phlox_assembly_collab/args_introgression/argweaver_analysis_feb2026/first_run/argweaver_runs/{window_filename}

mamba activate py312

/n/home09/pfmckenzie/pfmckenzie/pkgs/ARGweaver/bin/arg-sample --vcf {window_vcf_path} \
--region {chrom}:{start}-{end} \
--mask-cluster 2,5 \
--mutrate 1e-8 \
--maxtime 1000000 \
--compress-seq 10 \
--iters 5000 \
-o ./out
""".format(window_filename=window_filename, 
           window_path=window_vcf_path,
           results_dir=results_dir,
           chrom=chrom,
           start=start,
           end=end,
          )
    slurm_path = os.path.join(results_dir, "run.slurm")
    with open(slurm_path, 'w') as f:
        f.write(slurm_script)
    
    subprocess.run(["sbatch", str(slurm_path)], check=True)

Submitted batch job 61042218
Submitted batch job 61042220
Submitted batch job 61042222
Submitted batch job 61042224
Submitted batch job 61042225
Submitted batch job 61042231
Submitted batch job 61042233
Submitted batch job 61042238
Submitted batch job 61042242
Submitted batch job 61042245
Submitted batch job 61042247
Submitted batch job 61042248
Submitted batch job 61042252
Submitted batch job 61042253
Submitted batch job 61042254
Submitted batch job 61042255
Submitted batch job 61042256
Submitted batch job 61042258
Submitted batch job 61042261
Submitted batch job 61042262


61042255 and 61042258 stopped...

In [56]:
window_filenames[-5] # 55

'outlier_windows.shared.drum.1_1103500000_1103750000.rep1.DP_5.vcf.gz'

In [57]:
window_filenames[-3] # 58

'outlier_windows.shared.roem.1_1126000000_1126250000.rep1.DP_5.vcf.gz'

In [58]:
window_filenames

['outlier_windows.shared.drum.4_358250000_358500000.rep1.DP_5.vcf.gz',
 'outlier_windows.shared.drum.6_198750000_199000000.rep1.DP_5.vcf.gz',
 'outlier_windows.shared.roem.3_319000000_319250000.rep1.DP_5.vcf.gz',
 'outlier_windows.shared.drum.4_37250000_37500000.rep1.DP_5.vcf.gz',
 'outlier_windows.shared.drum.2_119250000_119500000.rep1.DP_5.vcf.gz',
 'outlier_windows.shared.roem.3_252000000_252250000.rep1.DP_5.vcf.gz',
 'outlier_windows.shared.roem.2_495250000_495500000.rep1.DP_5.vcf.gz',
 'outlier_windows.shared.drum.4_44250000_44500000.rep1.DP_5.vcf.gz',
 'outlier_windows.shared.drum.2_409250000_409500000.rep1.DP_5.vcf.gz',
 'outlier_windows.shared.roem.1_560750000_561000000.rep1.DP_5.vcf.gz',
 'outlier_windows.shared.roem.4_263750000_264000000.rep1.DP_5.vcf.gz',
 'outlier_windows.shared.roem.7_72750000_73000000.rep1.DP_5.vcf.gz',
 'outlier_windows.shared.roem.4_264000000_264250000.rep1.DP_5.vcf.gz',
 'outlier_windows.shared.roem.3_842250000_842500000.rep1.DP_5.vcf.gz',
 'outlier_wi

### I think argweaver must not be able to handle the really big numbers (it's failing on 1_1126000000_1126250000 and 1_1103500000_1103750000)

# Now do sample windows

In [79]:
windows_dir = "/n/holylfs05/LABS/hopkins_lab/Lab/PROJECTS/Phlox_assembly_collab/args_introgression/vcf_tarballs/results_20260210/results/samp_windows/"
window_filenames = os.listdir(windows_dir)
window_filenames = [i for i in window_filenames if i.endswith('.gz')] # getting rid of index files
window_filenames = [i for i in window_filenames if 'rep1' in i] # my choice... only doing one rep per window for now

In [80]:
len(window_filenames)

100

In [81]:
len(np.unique(window_filenames))

100

In [82]:
for window_filename in window_filenames:
    window_vcf_path = os.path.join(windows_dir,window_filename)
    
    results_dir = os.path.join('/n/home09/pfmckenzie/lab_lfs/Lab/PROJECTS/Phlox_assembly_collab/args_introgression/argweaver_analysis_feb2026/first_run/argweaver_runs/samp_windows',window_filename)
    os.makedirs(results_dir,exist_ok=True)

    chrom, start, end = window_filename.split('.')[1].split('_') # I changed this from outlier windows
    
    slurm_script = f"""#!/bin/sh
#SBATCH -c 4
#SBATCH -t 0-23:59:00
#SBATCH -p sapphire
#SBATCH --output={results_dir}/slurm-%j.out
#SBATCH --mem=20G
#SBATCH --job-name=aw

cd {results_dir}

mamba activate py312

/n/home09/pfmckenzie/pfmckenzie/pkgs/ARGweaver/bin/arg-sample --vcf {window_vcf_path} \
--region {chrom}:{start}-{end} \
--mask-cluster 2,5 \
--mutrate 1e-8 \
--maxtime 1000000 \
--compress-seq 10 \
--iters 5000 \
-o ./out
""".format(window_filename=window_filename, 
           window_path=window_vcf_path,
           results_dir=results_dir,
           chrom=chrom,
           start=start,
           end=end,
          )
    slurm_path = os.path.join(results_dir, "run.slurm")
    with open(slurm_path, 'w') as f:
        f.write(slurm_script)
    
    subprocess.run(["sbatch", str(slurm_path)], check=True)

Submitted batch job 61047250
Submitted batch job 61047251
Submitted batch job 61047252
Submitted batch job 61047255
Submitted batch job 61047256
Submitted batch job 61047259
Submitted batch job 61047260
Submitted batch job 61047261
Submitted batch job 61047263
Submitted batch job 61047264
Submitted batch job 61047265
Submitted batch job 61047266
Submitted batch job 61047267
Submitted batch job 61047268
Submitted batch job 61047269
Submitted batch job 61047270
Submitted batch job 61047271
Submitted batch job 61047274
Submitted batch job 61047275
Submitted batch job 61047276
Submitted batch job 61047277
Submitted batch job 61047278
Submitted batch job 61047279
Submitted batch job 61047280
Submitted batch job 61047281
Submitted batch job 61047282
Submitted batch job 61047285
Submitted batch job 61047286
Submitted batch job 61047288
Submitted batch job 61047289
Submitted batch job 61047290
Submitted batch job 61047291
Submitted batch job 61047292
Submitted batch job 61047294
Submitted batc